In [3]:
# Функции для нормализации даныых

def calculate_normalization_params(all_images):
    """
    Рассчитываем параметры нормализации по ВСЕМУ датасету
    """
    all_pixels = np.concatenate([img.flatten() for img in all_images])
    
    # Используем процентили как вы сказали (устойчиво к выбросам)
    p5 = np.percentile(all_pixels, 5)   # 5-й процентиль
    p95 = np.percentile(all_pixels, 95) # 95-й процентиль
    
    gamma = 2
    
    return {
        'p5': p5,
        'p95': p95, 
        'gamma': gamma,
        'range_min': np.min(all_pixels),  # на всякий случай
        'range_max': np.max(all_pixels)
    }

def normalize_image(image, params):
    """
    Нормализация отдельного изображения с сохраненными параметрами
    """
    p5, p95, gamma = params['p5'], params['p95'], params['gamma']
    
    # 1. Обрезаем выбросы и масштабируем
    image_clipped = np.clip(image, p5, p95)
    image_scaled = (image_clipped - p5) / (p95 - p5)  # [0, 1]
    
    # 2. Нелинейное преобразование (gamma=2 из статьи)
    image_transformed = image_scaled ** (1/gamma)
    
    # 3. В диапазон [-1, 1]
    image_normalized = 2 * image_transformed - 1
    
    return image_normalized

def denormalize_image(image_normalized, params):
    """
    Обратное преобразование
    """
    p5, p95, gamma = params['p5'], params['p95'], params['gamma']
    
    # 1. Из [-1, 1] в [0, 1]
    image_transformed = (image_normalized + 1) / 2
    
    # 2. Обратное нелинейное преобразование
    image_scaled = image_transformed ** gamma
    
    # 3. Обратно к исходному диапазону
    image_original = image_scaled * (p95 - p5) + p5
    
    return image_original

def normalize_dataset(dataset):
    if dataset.get("params", None) is not None:
        raise Exception("Dataset is normaliazed already")
    print("Подсчёт метрик")
    params = calculate_normalization_params(dataset["ACT"])
    print("Нормализация ACT")
    for i in tqdm(range(len(dataset["ACT"]))):
        dataset["ACT"][i] = normalize_image(dataset["ACT"][i], params)
    print("Нормализация Planck")
    for i in tqdm(range(len(dataset["Planck"]))):
        dataset["Planck"][i] = normalize_image(dataset["Planck"][i], params)
    dataset["params"] = params

In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import cv2
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits
import astropy.units as u
import matplotlib.patheffects as path_effects
from PIL import Image
from functools import lru_cache 

def load_dataset_fits(filename):
    """Загружает датасет из FITS файла"""
    
    with fits.open(filename) as hdul:
        
        dataset = {}
        
        # Читаем метаданные из primary HDU
        header = hdul[0].header
        metadata = {
            'diameter_arcmin': header.get('DIAMETER', 16),
            'pixscale_arcmin': header.get('PIXSCALE', 0.5),
            'patch_shape': (header.get('SHAPE0', 32), header.get('SHAPE1', 32))
        }
        
        # Загружаем центры
        centers_hdu = hdul['CENTERS']
        centers_data = centers_hdu.data
        centers = list(zip(centers_data['RA'], centers_data['DEC']))
        metadata['centers'] = centers
        
        # Проверяем структуру файла
        if 'ACT_PATCHES' in hdul and 'PLANCK_PATCHES' in hdul:
            # Оптимизированный формат
            act_cube = hdul['ACT_PATCHES'].data
            planck_cube = hdul['PLANCK_PATCHES'].data
            
            dataset['ACT'] = [act_cube[i] for i in range(act_cube.shape[0])]
            dataset['Planck'] = [planck_cube[i] for i in range(planck_cube.shape[0])]
            
        else:
            # Поиск отдельных расширений
            act_patches = []
            planck_patches = []
            
            for hdu in hdul:
                if hdu.name.startswith('ACT_'):
                    act_patches.append(hdu.data)
                elif hdu.name.startswith('PLANCK_'):
                    planck_patches.append(hdu.data)
            
            # Сортируем по имени чтобы сохранить порядок
            dataset['ACT'] = act_patches
            dataset['Planck'] = planck_patches
        
        dataset['metadata'] = metadata
        return dataset

# dataset_fits = {}
    
def get_image(index=0):
    """Визуализирует одну пару ACT/Planck патчей"""
    
    centers = dataset_fits['metadata']['centers']
    
    if index >= len(centers):
        print(f"Ошибка: индекс {index} превышает количество центров ({len(centers)})")
        return
    
    ra, dec = centers[index]
        
    # ACT патч
    act_patch = dataset_fits['ACT'][index]
    planck_patch = dataset_fits['Planck'][index]
    
    # Convert to numpy array
    array2 = np.array(act_patch)
    array = np.array(planck_patch)
    # print(normalize_image(array, calculate_normalization_params(array)))
    # print(array)

    return planck_patch, act_patch
    # print(normalize_image(array, calculate_normalization_params(array)))
    # print(array)
    # raise Exception()
    # return (normalize_image(array, calculate_normalization_params(array)), 
            # normalize_image(array2, calculate_normalization_params(array2)))

def get_arrays():
    act_patch = dataset_fits['ACT']
    planck_patch = dataset_fits['Planck']
    
    return (normalize_image(array, calculate_normalization_params(planck_patch)), 
            normalize_image(array2, calculate_normalization_params(act_patch)))
    
class ImageDataset(Dataset):
    def __init__(self, transform=None, source_data=None):
        self.transform = transform
        self.upscale = 1
        self.source_data = source_data
        
    def __len__(self):
        return len(self.source_data['metadata']['centers'])
    
    def __getitem__(self, idx):        
        low_res, image = get_image(index=idx)
        # print(image)
        if self.transform:
            image = self.transform(image)
            low_res = self.transform(low_res)
            
        return low_res, image

class UpsampleModel(nn.Module):
    def __init__(self):
        super(UpsampleModel, self).__init__()
        # Encoder part
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        
        # self.upsample1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        # self.upsample2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        # self.upsample3 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        
        self.conv_final = nn.Conv2d(256, 3, kernel_size=3, padding=1)
        
    def forward(self, x):
        # Encoder
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        
        # Decoder
        x = torch.tanh(self.conv_final(x))
        
        return x

import torch
from torchvision import transforms
from PIL import Image
import numpy as np

def pil_list_to_tensor(pil_images):
    """
    Convert list of PIL Images to tensor with shape (len, 1, width, height)
    
    Args:
        pil_images: List of PIL.Image objects
    
    Returns:
        torch.Tensor with shape (len, 1, width, height)
    """
    # Convert each PIL image to tensor and add channel dimension
    tensors = []
    for img in pil_images:
        # Convert PIL to numpy array
        np_img = np.array(img)
        
        # Add channel dimension if grayscale (H, W) -> (1, H, W)
        if len(np_img.shape) == 2:
            np_img = np.expand_dims(np_img, axis=0)
        # If RGB, convert to grayscale and add channel dimension
        elif len(np_img.shape) == 3:
            np_img = np_img.mean(axis=2, keepdims=True)
            np_img = np_img.transpose(2, 0, 1)  # (H, W, 1) -> (1, H, W)
        
        # Convert to tensor
        tensor = torch.from_numpy(np_img).float()
        tensors.append(tensor)
    
    # Stack all tensors along the first dimension
    return torch.stack(tensors)

    


In [31]:
# dataset_fits = load_dataset_fits("/home/jupyter/datasphere/filestore/datasets/3_mask_dataset.fits")
normalize_dataset(dataset_fits)
print(dataset_fits.get('params', 0))

Подсчёт метрик
Нормализация ACT


100%|██████████| 24718/24718 [00:07<00:00, 3470.53it/s]


Нормализация Planck


100%|██████████| 24718/24718 [01:01<00:00, 398.80it/s]

{'p5': -8.774256678786559e-06, 'p95': 8.496879074122443e-06, 'gamma': 2, 'range_min': -6.25698122077788e-05, 'range_max': 0.00022538719647646833}


In [33]:
from torchvision import transforms
transform = transforms.Compose([ 
 transforms.ToTensor() 
])
dataset = ImageDataset(transform=transform, source_data=dataset_fits)
# print(dataset_fits['ACT'][0])
# print(dataset_fits['params'])
dataset[0]

(tensor([[[0.2163, 0.2218, 0.2275,  ..., 0.5793, 0.5804, 0.5810],
          [0.2181, 0.2237, 0.2300,  ..., 0.5780, 0.5808, 0.5823],
          [0.2225, 0.2281, 0.2352,  ..., 0.5751, 0.5803, 0.5826],
          ...,
          [0.5420, 0.5507, 0.5593,  ..., 0.3351, 0.3291, 0.3232],
          [0.5413, 0.5505, 0.5596,  ..., 0.3196, 0.3135, 0.3074],
          [0.5381, 0.5478, 0.5575,  ..., 0.3051, 0.2981, 0.2913]]],
        dtype=torch.float64),
 tensor([[[-0.3057, -0.2355, -0.0719,  ...,  0.3856,  0.3703,  0.3472],
          [ 0.3829,  0.3045,  0.2833,  ...,  0.4639,  0.7744,  0.8692],
          [ 0.3694,  0.3938,  0.4084,  ...,  0.7744,  0.9350,  0.9172],
          ...,
          [ 0.4158,  0.4166,  0.4174,  ...,  0.4268,  0.4041,  0.3689],
          [ 0.4183,  0.4175,  0.4175,  ...,  0.4125,  0.4196,  0.4225],
          [ 0.4167,  0.4161,  0.4156,  ...,  0.4094,  0.4109,  0.4171]]],
        dtype=torch.float64))

In [80]:
dataset_fits2 = load_dataset_fits("/home/jupyter/datasphere/filestore/datasets/mask_dataset.fits")

In [107]:
print(np.max(dataset_fits2['ACT'][:10]))
print(np.max(dataset_fits['ACT'][:10]))
# dataset_fits["metadata"]

2.685439549740741e-05
1.0


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from einops import rearrange

class GaussianDiffusion:
    def __init__(self, timesteps=1000, beta_schedule='linear'):
        self.timesteps = timesteps
        
        if beta_schedule == 'linear':
            self.betas = torch.linspace(1e-4, 0.02, timesteps)
        else:
            self.betas = torch.linspace(1e-4, 0.02, timesteps)
            
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
    
    def add_noise(self, x_start, t, noise, device):
        if noise is None:
            noise = torch.randn_like(x_start)
            
        sqrt_alpha_cumprod = self.sqrt_alphas_cumprod.to(device)[t.to(device)].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_alphas_cumprod.to(device)[t].view(-1, 1, 1, 1)
        
        return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha_cumprod * noise
    
    def sample_timesteps(self, n):
        return torch.randint(low=1, high=self.timesteps, size=(n,))

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=64):
        super().__init__()
        
        self.encoder1 = self._block(in_channels, base_channels)
        self.encoder2 = self._block(base_channels, base_channels * 2)
        self.encoder3 = self._block(base_channels * 2, base_channels * 4)
        
        self.bottleneck = self._block(base_channels * 4, base_channels * 8)
        
        self.decoder3 = self._block(base_channels * 12, base_channels * 4)  # skip connection
        self.decoder2 = self._block(base_channels * 6, base_channels * 2)   # skip connection
        self.decoder1 = self._block(base_channels * 3, base_channels)       # skip connection
        
        self.final_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)
        
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
    def _block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x, time_emb=None):
        # Encoder
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool(e3))
        
        # Decoder with skip connections
        d3 = self.decoder3(torch.cat([self.upsample(bottleneck), e3], dim=1))
        d2 = self.decoder2(torch.cat([self.upsample(d3), e2], dim=1))
        d1 = self.decoder1(torch.cat([self.upsample(d2), e1], dim=1))
        
        return self.final_conv(d1)

import torch
import torch.nn.functional as F
from torchmetrics import StructuralSimilarityIndexMeasure

def mse_loss(pred, target):
    return F.mse_loss(pred, target)

def psnr(pred, target, data_range=1.0):
    mse = F.mse_loss(pred, target)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(data_range / torch.sqrt(mse))

ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0, reduction='elementwise_mean')

class SuperResolutionDiffusion:
    def __init__(self, model, diffusion, lr_size=32, hr_size=128, device='cuda'):
        self.model = model.to(device)
        self.diffusion = diffusion
        self.lr_size = lr_size
        self.hr_size = hr_size
        self.device = device
        
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        
        self.total_mse = 0.0
        self.total_psnr = 0.0
        self.total_ssim = 0.0
        self.n_batches = 0
        
    def train_step(self, lr_imgs, hr_imgs, device):
        self.model.train()
        
        # Подготовка данных
        # lr_imgs = F.interpolate(lr_imgs, size=self.hr_size, mode='bilinear')
        batch_size = lr_imgs.shape[0]
        
        # Случайные временные шаги
        t = self.diffusion.sample_timesteps(batch_size).to(self.device)
        
        # Добавление шума к HR изображениям
        noise = torch.randn_like(hr_imgs)
        noisy_imgs = self.diffusion.add_noise(hr_imgs, t, noise, device)
        
        # Конкатенация LR и зашумленных HR изображений
        model_input = torch.cat([lr_imgs, noisy_imgs], dim=1)
        
        # Предсказание шума
        predicted_noise = self.model(model_input)
        
        # Loss function
        loss = F.mse_loss(predicted_noise, noise)
        
        batch_mse = F.mse_loss(predicted_noise, noise, reduction='mean')
        batch_psnr = psnr(predicted_noise, noise)
        batch_ssim = ssim_metric.to(device)(predicted_noise, noise)

        self.total_mse += batch_mse.item()
        self.total_psnr += batch_psnr.item()
        self.total_ssim += batch_ssim.item()
        self.n_batches += 1
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return {
            'mse': self.total_mse / max(1, self.n_batches),
            'psnr': self.total_psnr / max(1, self.n_batches),
            'ssim': self.total_ssim / max(1, self.n_batches),
        }
    
    @torch.no_grad()
    def sample(self, lr_imgs, num_samples, device, t=30):
        self.model.eval()
        
        # Интерполяция LR до HR размера
        # lr_upscaled = F.interpolate(lr_imgs, size=self.hr_size, mode='bilinear')
        lr_upscaled = lr_imgs
        
        # Начинаем с шума
        x = torch.randn(num_samples, 1, self.hr_size, self.hr_size).to(self.device)
        
        # Обратный диффузионный процесс
        print(self.diffusion.timesteps)
        # self.diffusion.timesteps = 30
        for i in reversed(range(t)):
            t = torch.full((num_samples,), i, device=self.device, dtype=torch.long)
            
            # Конкатенация с LR изображением
            model_input = torch.cat([lr_upscaled, x], dim=1)
            
            # Предсказание шума
            predicted_noise = self.model(model_input)
            
            # Коэффициенты для шага денойзинга
            alpha = self.diffusion.alphas.to(device)[t].view(-1, 1, 1, 1)
            alpha_cumprod = self.diffusion.alphas_cumprod.to(device)[t].view(-1, 1, 1, 1)
            beta = self.diffusion.betas.to(device)[t].view(-1, 1, 1, 1)
            
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
                
            # Шаг денойзинга
            x = (1 / torch.sqrt(alpha)) * (
                x - ((1 - alpha) / (torch.sqrt(1 - alpha_cumprod))) * predicted_noise
            ) + torch.sqrt(beta) * noise
        
        return x
    
    @torch.no_grad()
    def short_sample(self, lr_imgs, num_samples, device, t=30):
        self.model.eval()
        
        # Интерполяция LR до HR размера
        # lr_upscaled = F.interpolate(lr_imgs, size=self.hr_size, mode='bilinear')
        lr_upscaled = lr_imgs
        
        # Начинаем с шума
        x = torch.randn(num_samples, 1, self.hr_size, self.hr_size).to(self.device)
        x = lr_imgs
        
        # Обратный диффузионный процесс
        # print(self.diffusion.timesteps)
        # self.diffusion.timesteps = 30
        for i in reversed(range(t)):
            t = torch.full((num_samples,), i, device=self.device, dtype=torch.long)
            
            # Конкатенация с LR изображением
            model_input = torch.cat([lr_upscaled, x], dim=1)
            
            # Предсказание шума
            predicted_noise = self.model(model_input)
            
            # Коэффициенты для шага денойзинга
            alpha = self.diffusion.alphas.to(device)[t].view(-1, 1, 1, 1)
            alpha_cumprod = self.diffusion.alphas_cumprod.to(device)[t].view(-1, 1, 1, 1)
            beta = self.diffusion.betas.to(device)[t].view(-1, 1, 1, 1)
            
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
                
            # Шаг денойзинга
            x = (1 / torch.sqrt(alpha)) * (
                x - ((1 - alpha) / (torch.sqrt(1 - alpha_cumprod))) * predicted_noise
            ) + torch.sqrt(beta) * noise
        
        return x
    @torch.no_grad()
    def fast_validation_sample(self, lr_imgs, device, sampling_steps=10):
        """Ускоренный sampling для валидации"""
        self.model.eval()

        lr_upscaled = lr_imgs
        x = torch.randn_like(lr_upscaled)  # шум того же размера

        # Используем большие шаги для ускорения
        step_size = self.diffusion.timesteps // sampling_steps
        timesteps = list(reversed(range(0, self.diffusion.timesteps, step_size)))

        for i, t in enumerate(timesteps):
            t_batch = torch.full((lr_imgs.size(0),), t, device=device, dtype=torch.long)

            model_input = torch.cat([lr_upscaled, x], dim=1)
            predicted_noise = self.model(model_input)

            alpha = self.diffusion.alphas.to(device)[t_batch].view(-1, 1, 1, 1)
            alpha_cumprod = self.diffusion.alphas_cumprod.to(device)[t_batch].view(-1, 1, 1, 1)
            beta = self.diffusion.betas.to(device)[t_batch].view(-1, 1, 1, 1)

            if i < len(timesteps) - 1:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)

            x = (1 / torch.sqrt(alpha)) * (
                x - ((1 - alpha) / (torch.sqrt(1 - alpha_cumprod))) * predicted_noise
            ) + torch.sqrt(beta) * noise

        return torch.clamp(x, -1.0, 1.0)  # клиппинг в диапазон
    def validate_fast(self, dataloader, device, num_batches=5, sampling_steps=10):
        """Быстрая валидация на нескольких батчах"""
        self.model.eval()

        total_mse, total_psnr, total_ssim = 0.0, 0.0, 0.0
        count = 0
        ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
        with torch.no_grad():
            for i, (lr_imgs, hr_imgs) in enumerate(dataloader):
                if i >= num_batches:  # только несколько батчей
                    break

                lr_imgs = lr_imgs.to(device).float()
                hr_imgs = hr_imgs.to(device).float()

                # Ускоренный sampling
                generated_hr = self.fast_validation_sample(lr_imgs, device, sampling_steps)

                # Метрики на GPU (быстрее)
                batch_mse = F.mse_loss(generated_hr, hr_imgs)
                batch_psnr = psnr(generated_hr, hr_imgs)
                batch_ssim = ssim_metric(generated_hr, hr_imgs)

                total_mse += batch_mse.item()
                total_psnr += batch_psnr.item() 
                total_ssim += batch_ssim.item()
                count += 1

        return {
            'val_mse': total_mse / count,
            'val_psnr': total_psnr / count,
            'val_ssim': total_ssim / count
        }

# def main():

# Параметры
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
hr_size = 128
lr_size = 128, 128
num_epochs = 20

# # Данные
# from torchvision import transforms
# transform = transforms.Compose([ 
#  transforms.ToTensor() 
# ])

# dataset_fits = load_dataset_fits("/home/jupyter/datasphere/filestore/datasets/mask_dataset.fits")
# normalize_dataset(dataset_fits)
# dataset = ImageDataset(transform=transform)
# dataset = CMB_Dataset(1000, 128, 128)


batch_size = 30

# Инициализация компонентов
diffusion = GaussianDiffusion(timesteps=50)
model = UNet(in_channels=2, out_channels=1).to(device)  # 2 канала: LR + noisy HR

sr_diffusion = SuperResolutionDiffusion(
    model, diffusion, lr_size, hr_size, device
)


dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Обучение
for epoch in range(num_epochs):
    # Валидация и визуализация
    # print("Validation")
    # val_metrics = sr_diffusion.validate_fast(dataloader, device, 
    #                    num_batches=3, sampling_steps=20)
    # print(f"Epoch {epoch} Validation: {val_metrics}")
    for i, (lr_imgs, hr_imgs) in enumerate(dataloader):
        lr_imgs = lr_imgs.to(device).float()
        hr_imgs = hr_imgs.to(device).float()

        total = sr_diffusion.train_step(lr_imgs, hr_imgs, device)

        if i % 100 == 0:
            print(f'Epoch {epoch}, Batch {i}, Loss: {total}')

    # Валидация и визуализация
    print("Validation")
    val_metrics = sr_diffusion.validate_fast(dataloader, device, 
                       num_batches=3, sampling_steps=20)
    print(f"Epoch {epoch} Validation: {val_metrics}")
#     with torch.no_grad():
#         m = p = s = 0
#         l = int(len(dataset) * 0.05)
#         for index in range(l):
#             # Тест на примере из валидации
#             test_lr, test_hr = dataset[index]
#             # left top rect


#             test_lr = test_lr.unsqueeze(0).to(device).float()
#             test_hr = test_hr.float()

#             # Генерация HR
#             generated_hr = sr_diffusion.short_sample(test_lr, 1, device, t).to('cpu').numpy()[0, 0]
#             generated_hr = normalize_image(generated_hr, calculate_normalization_params(generated_hr))
#             test_hr = test_hr.numpy()[0]
#             # generated_hr = normalize_image(generated_hr.cpu(), calculate_normalization_params(generated_hr.cpu())).to(device)
#             m += calculate_mse(test_hr, generated_hr)
#             p += calculate_psnr(test_hr, generated_hr)
#             s += calculate_ssim(test_hr, generated_hr)
#         print(f"MSE={m/l}")
#         print(f"PSNR={p/l}")
#         print(f"SSIM={s/l}")
#         # Тест на примере из валидации
#         test_lr, test_hr = dataset[0]
#         test_lr = test_lr.unsqueeze(0).to(device).float()
#         test_hr = test_hr.float()
#         # Генерация HR
#         generated_hr = sr_diffusion.sample(test_lr, 1, device, 1000)
#         # generated_hr = normalize_image(generated_hr.cpu(), calculate_normalization_params(generated_hr.cpu())).to(device)

#         # Визуализация
#         plt.figure(figsize=(15, 5))

#         plt.subplot(1, 3, 1)
#         plt.imshow(test_lr[0, 0].cpu().numpy(), cmap='viridis')
#         plt.title('LR Input')
#         plt.colorbar()

#         plt.subplot(1, 3, 2)
#         plt.imshow(generated_hr[0, 0].cpu().numpy(), cmap='viridis')
#         plt.title('Generated HR')
#         plt.colorbar()

#         plt.subplot(1, 3, 3)
#         plt.imshow(test_hr[0].cpu().numpy(), cmap='viridis')
#         plt.title('True HR')
#         plt.colorbar()

#         plt.tight_layout()
#         plt.savefig(f'results_epoch_{epoch}.png')
#         plt.close()

    # Сохранение модели
    torch.save(model.state_dict(), f'sr_diffusion_model_epoch_{epoch}.pth')


cuda
Epoch 0, Batch 0, Loss: {'mse': 1.0541679859161377, 'psnr': -0.22909852862358093, 'ssim': 0.005706945899873972}


KeyboardInterrupt: 

In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

hr_size = 128
lr_size = 128, 128

model = UNet(in_channels=2, out_channels=1).to(device)
model.load_state_dict(torch.load('/home/jupyter/datasphere/project/sr_diffusion_model_epoch_19.pth'))
diffusion = GaussianDiffusion(timesteps=1000)
sr_diffusion = SuperResolutionDiffusion(
        model, diffusion, lr_size, hr_size, device
    )

# dataset = CMB_Dataset(1000, 128, 128)
# from torchvision import transforms
# transform = transforms.Compose([ 
 # transforms.ToTensor() 
# ])

# dataset_fits = load_dataset_fits("/home/jupyter/datasphere/filestore/datasets/64minutes_dataset.fits")
# dataset = ImageDataset(transform=transform)

for index in range(10):
    with torch.no_grad():
        # Тест на примере из валидации
        test_lr, test_hr = dataset[index]
        # left top rect
        
        
        test_lr = test_lr.unsqueeze(0).to(device).float()
        test_hr = test_hr.float()
        
        # Генерация HR
        generated_hr = sr_diffusion.short_sample(test_lr, 1, device, 100).to('cpu').numpy()[0, 0]
        test_hr = test_hr.numpy()[0]
        # generated_hr = normalize_image(generated_hr.cpu(), calculate_normalization_params(generated_hr.cpu())).to(device)
        print(calculate_mse(test_hr, generated_hr), calculate_psnr(test_hr, generated_hr), calculate_ssim(test_hr, generated_hr))
        
        # Визуализация
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.imshow(test_lr[0, 0].cpu().numpy(), cmap='viridis', vmin=-1, vmax=1)
        plt.title('LR Input')
        plt.colorbar()

        plt.subplot(1, 3, 2)
        # plt.imshow(generated_hr[0, 0].cpu().numpy(), cmap='viridis', vmin=-1, vmax=1)
        plt.imshow(generated_hr, cmap='viridis', vmin=-1, vmax=1)
        plt.title('Generated HR')
        plt.colorbar()

        plt.subplot(1, 3, 3)
        plt.imshow(test_hr, cmap='viridis', vmin=-1, vmax=1)
        plt.title('True HR')
        plt.colorbar()

        plt.tight_layout()
        plt.savefig(f'results/results_test_{index}.png')
        plt.close()
    print('end', index)


0.045187563 13.44981074333191 0.2904442560217633
end 0
0.10179874 9.922575950622559 0.0682508924977442
end 1
0.15370066 8.133242726325989 0.04319646477791399
end 2
0.14034496 8.528032302856445 0.050641193775636056
end 3
0.16019738 7.953445911407471 0.04356590396232091
end 4
0.18912171 7.232586145401001 0.03319937194289309
end 5
0.17902713 7.470811605453491 0.03664449547870566
end 6
0.18877095 7.24064826965332 0.03991629704226497
end 7
0.16118316 7.926803231239319 0.043743342919788826
end 8
0.14987765 8.242630958557129 0.0428794657050489
end 9


In [155]:
for t in [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    m = p = s = 0
    for index in range(40):
        with torch.no_grad():
            # Тест на примере из валидации
            test_lr, test_hr = dataset[index]
            # left top rect


            test_lr = test_lr.unsqueeze(0).to(device).float()
            test_hr = test_hr.float()

            # Генерация HR
            generated_hr = sr_diffusion.short_sample(test_lr, 1, device, t).to('cpu').numpy()[0, 0]
            # generated_hr = normalize_image(generated_hr, calculate_normalization_params(generated_hr))
            test_hr = test_hr.numpy()[0]
            # generated_hr = normalize_image(generated_hr.cpu(), calculate_normalization_params(generated_hr.cpu())).to(device)
            m += calculate_mse(test_hr, generated_hr)
            p += calculate_psnr(test_hr, generated_hr)
            s += calculate_ssim(test_hr, generated_hr)
    print(t, m / 40, p / 40, s / 40)
        

5 0.1484053522348404 8.513731449842453 0.06621044039831828
10 0.14919538665562868 8.492015734314919 0.06508236946160362
20 0.15123459007591009 8.436797812581062 0.06291567789465088
30 0.15328375361859797 8.372648432850838 0.062122658227042785
40 0.15571193573996425 8.302048295736313 0.06100971998318742
50 0.1592976490035653 8.190041989088058 0.06066503838774194


../aten/src/ATen/native/cuda/IndexKernel.cu:92: operator(): block: [0,0,0], thread: [0,0,0] Assertion `index >= -sizes[i] && index < sizes[i] && "index out of bounds"` failed.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [12]:
from skimage.metrics import structural_similarity as ssim
def calculate_mse(img_true, img_pred):
    return np.mean((img_true - img_pred) ** 2)

def calculate_psnr(img_true, img_pred):
    mse = calculate_mse(img_true, img_pred)
    if mse == 0:
        return float('inf')
    max_pixel = np.max(img_true)
    psnr = 20 * np.log10(max_pixel) - 10 * np.log10(mse)
    return psnr

def calculate_ssim(img_true, img_pred):
    # Используется реализация из skimage как в статье
    return ssim(img_true, img_pred, 
                data_range=img_true.max()-img_true.min(),
                c1=1e-4, c2=9e-4)  # константы из статьи

test_lr shape: torch.Size([1, 128, 128])
test_lr dtype: torch.float64
test_lr min/max: 0.17960930620154936, 0.9703847291890222
Device: cuda
NaN in test_lr: False
Inf in test_lr: False


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, base_channels=64, time_emb_dim=32):
        super().__init__()
        self.time_emb_dim = time_emb_dim
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, base_channels * 4),
            nn.ReLU(),
            nn.Linear(base_channels * 4, base_channels * 4)
        )
        
        # Encoder
        self.encoder1 = self._block(in_channels, base_channels)
        self.encoder2 = self._block(base_channels, base_channels * 2)
        self.encoder3 = self._block(base_channels * 2, base_channels * 4)
        
        # Bottleneck
        self.bottleneck = self._block(base_channels * 4, base_channels * 8)
        
        # Decoder with skip connections
        self.decoder3 = self._block(base_channels * 12, base_channels * 4)
        self.decoder2 = self._block(base_channels * 6, base_channels * 2)
        self.decoder1 = self._block(base_channels * 3, base_channels)
        
        self.final_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)
        
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
    def _block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x, time_emb=None):
        # Process time embedding
        if time_emb is not None:
            time_emb = self.time_mlp(time_emb)
            time_emb = time_emb.view(-1, time_emb.shape[1], 1, 1)
        
        # Encoder
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        
        # Bottleneck with time embedding
        bottleneck_input = self.pool(e3)
        if time_emb is not None:
            # Add time embedding to bottleneck
            time_emb_expanded = time_emb.expand(-1, -1, bottleneck_input.shape[2], bottleneck_input.shape[3])
            bottleneck_input = bottleneck_input + time_emb_expanded[:, :bottleneck_input.shape[1]]
        
        bottleneck = self.bottleneck(bottleneck_input)
        
        # Decoder with skip connections
        d3 = self.decoder3(torch.cat([self.upsample(bottleneck), e3], dim=1))
        d2 = self.decoder2(torch.cat([self.upsample(d3), e2], dim=1))
        d1 = self.decoder1(torch.cat([self.upsample(d2), e1], dim=1))
        
        return self.final_conv(d1)

class ImprovedGaussianDiffusion:
    def __init__(self, timesteps=1000, beta_schedule='linear'):
        self.timesteps = timesteps
        
        if beta_schedule == 'linear':
            beta_start = 1e-4
            beta_end = 0.02
            self.betas = torch.linspace(beta_start, beta_end, timesteps)
        else:
            self.betas = torch.linspace(1e-4, 0.02, timesteps)
            
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        
        # Calculations for diffusion q(x_t | x_{t-1})
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        self.log_one_minus_alphas_cumprod = torch.log(1. - self.alphas_cumprod)
        self.sqrt_recip_alphas_cumprod = torch.sqrt(1. / self.alphas_cumprod)
        self.sqrt_recipm1_alphas_cumprod = torch.sqrt(1. / self.alphas_cumprod - 1)
        
        # Calculations for posterior q(x_{t-1} | x_t, x_0)
        self.posterior_variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)
        
    def add_noise(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
            
        sqrt_alpha_cumprod = self.sqrt_alphas_cumprod.to(x_start.device)[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_alphas_cumprod.to(x_start.device)[t].view(-1, 1, 1, 1)
        
        return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha_cumprod * noise
    
    def sample_timesteps(self, n, device):
        return torch.randint(low=0, high=self.timesteps, size=(n,), device=device)

class ImprovedSuperResolutionDiffusion:
    def __init__(self, model, diffusion, device='cuda'):
        self.model = model.to(device)
        self.diffusion = diffusion
        self.device = device
        
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=1000)
        
    def get_time_embedding(self, timesteps, dim=32):
        """Sinusoidal time embedding"""
        half_dim = dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=self.device) * -emb)
        emb = timesteps[:, None].float() * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        if dim % 2 == 1:  # zero pad
            emb = F.pad(emb, (0, 1))
        return emb
    
    def train_step(self, lr_imgs, hr_imgs):
        self.model.train()
        batch_size = lr_imgs.shape[0]
        
        # Sample timesteps
        t = self.diffusion.sample_timesteps(batch_size, self.device)
        
        # Add noise to HR images
        noise = torch.randn_like(hr_imgs)
        noisy_imgs = self.diffusion.add_noise(hr_imgs, t, noise)
        
        # Get time embeddings
        time_emb = self.get_time_embedding(t)
        
        # Concatenate LR and noisy HR
        model_input = torch.cat([lr_imgs, noisy_imgs], dim=1)
        
        # Predict noise
        predicted_noise = self.model(model_input, time_emb)
        
        # Simple MSE loss
        loss = F.mse_loss(predicted_noise, noise)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        self.scheduler.step()
        
        return loss.item()
    
    @torch.no_grad()
    def sample(self, lr_imgs, sampling_steps=50):
        self.model.eval()
        
        batch_size = lr_imgs.shape[0]
        img_size = lr_imgs.shape[2]  # assuming square images
        
        # Start from noise
        x = torch.randn(batch_size, 1, img_size, img_size, device=self.device)
        
        # Sampling steps
        step_size = self.diffusion.timesteps // sampling_steps
        timesteps = list(reversed(range(0, self.diffusion.timesteps, step_size)))
        
        for i, t in enumerate(timesteps):
            t_batch = torch.full((batch_size,), t, device=self.device, dtype=torch.long)
            time_emb = self.get_time_embedding(t_batch)
            
            # Model input
            model_input = torch.cat([lr_imgs, x], dim=1)
            predicted_noise = self.model(model_input, time_emb)
            
            # Diffusion parameters
            alpha = self.diffusion.alphas.to(self.device)[t_batch].view(-1, 1, 1, 1)
            alpha_cumprod = self.diffusion.alphas_cumprod.to(self.device)[t_batch].view(-1, 1, 1, 1)
            beta = self.diffusion.betas.to(self.device)[t_batch].view(-1, 1, 1, 1)
            
            if t > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
            
            # Denoising step (DDPM)
            x = (1 / torch.sqrt(alpha)) * (
                x - ((1 - alpha) / torch.sqrt(1 - alpha_cumprod)) * predicted_noise
            ) + torch.sqrt(beta) * noise
        
        return torch.clamp(x, -1.0, 1.0)

In [16]:
# Использование
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Инициализация с улучшенными параметрами
diffusion = ImprovedGaussianDiffusion(timesteps=1000)
model = ImprovedUNet(in_channels=2, out_channels=1, base_channels=64, time_emb_dim=32)
sr_diffusion = ImprovedSuperResolutionDiffusion(model, diffusion, device)
num_epochs = 20
# Обучение с мониторингом
for epoch in range(num_epochs):
    epoch_loss = 0
    for i, (lr_imgs, hr_imgs) in enumerate(dataloader):
        lr_imgs = lr_imgs.to(device).float()
        hr_imgs = hr_imgs.to(device).float()
        
        # Нормализация в диапазон [-1, 1]
        lr_imgs = (lr_imgs - 0.5) * 2
        hr_imgs = (hr_imgs - 0.5) * 2
        
        loss = sr_diffusion.train_step(lr_imgs, hr_imgs)
        epoch_loss += loss
        
        if i % 100 == 0:
            print(f'Epoch {epoch}, Batch {i}, Loss: {loss:.4f}')
    
    print(f'Epoch {epoch} Average Loss: {epoch_loss/len(dataloader):.4f}')
    
    # Валидация
    if epoch % 1 == 0:
        with torch.no_grad():
            # Тестирование на нескольких примерах
            test_lr, test_hr = next(iter(dataloader))
            test_lr = test_lr[:4].to(device).float()
            test_hr = test_hr[:4].to(device).float()
            
            test_lr = (test_lr - 0.5) * 2
            test_hr = (test_hr - 0.5) * 2
            
            generated = sr_diffusion.sample(test_lr, sampling_steps=50)
            
            # Денормализация для визуализации
            generated = (generated + 1) / 2
            test_lr_vis = (test_lr + 1) / 2
            test_hr_vis = (test_hr + 1) / 2
            
            # Визуализация результатов
            fig, axes = plt.subplots(4, 3, figsize=(12, 16))
            for idx in range(4):
                axes[idx, 0].imshow(test_lr_vis[idx, 0].cpu(), cmap='viridis')
                axes[idx, 0].set_title('LR Input')
                axes[idx, 1].imshow(generated[idx, 0].cpu(), cmap='viridis')
                axes[idx, 1].set_title('Generated HR')
                axes[idx, 2].imshow(test_hr_vis[idx, 0].cpu(), cmap='viridis')
                axes[idx, 2].set_title('True HR')
            
            plt.tight_layout()
            plt.savefig(f'improved_results_epoch_{epoch}.png')
            plt.close()

Epoch 0, Batch 0, Loss: 1.2047
Epoch 0, Batch 100, Loss: 0.1557
Epoch 0, Batch 200, Loss: 0.0891
Epoch 0, Batch 300, Loss: 0.1685
Epoch 0, Batch 400, Loss: 0.1426
Epoch 0, Batch 500, Loss: 0.1499
Epoch 0, Batch 600, Loss: 0.1924
Epoch 0, Batch 700, Loss: 0.1384
Epoch 0 Average Loss: 0.1610
Epoch 1, Batch 0, Loss: 0.2042
Epoch 1, Batch 100, Loss: 0.1434
Epoch 1, Batch 200, Loss: 0.1935
Epoch 1, Batch 300, Loss: 0.1630
Epoch 1, Batch 400, Loss: 0.1249
Epoch 1, Batch 500, Loss: 0.1926
Epoch 1, Batch 600, Loss: 0.2107
Epoch 1, Batch 700, Loss: 0.1236
Epoch 1 Average Loss: 0.1420
Epoch 2, Batch 0, Loss: 0.1831
Epoch 2, Batch 100, Loss: 0.1347
Epoch 2, Batch 200, Loss: 0.1642
Epoch 2, Batch 300, Loss: 0.1376
Epoch 2, Batch 400, Loss: 0.1270
Epoch 2, Batch 500, Loss: 0.0823
Epoch 2, Batch 600, Loss: 0.1432
Epoch 2, Batch 700, Loss: 0.1083
Epoch 2 Average Loss: 0.1394
Epoch 3, Batch 0, Loss: 0.1358
Epoch 3, Batch 100, Loss: 0.1672
Epoch 3, Batch 200, Loss: 0.2135
Epoch 3, Batch 300, Loss: 0.08

KeyboardInterrupt: 

In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

class CorrectSuperResolutionUNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, base_channels=64, time_emb_dim=32):
        super().__init__()
        
        self.time_emb_dim = time_emb_dim
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, base_channels * 4),
            nn.SiLU(),
            nn.Linear(base_channels * 4, base_channels * 4)
        )
        
        # Encoder (работает с конкатенированными LR + noisy HR)
        self.enc1 = self._block(in_channels, base_channels)  # 2 входных канала!
        self.enc2 = self._block(base_channels, base_channels * 2)
        self.enc3 = self._block(base_channels * 2, base_channels * 4)
        
        # Bottleneck
        self.bottleneck = self._block(base_channels * 4, base_channels * 8)
        
        # Decoder (генерирует шум для HR)
        self.dec3 = self._block(base_channels * 8 + base_channels * 4, base_channels * 4)  # skip from enc3
        self.dec2 = self._block(base_channels * 4 + base_channels * 2, base_channels * 2)   # skip from enc2  
        self.dec1 = self._block(base_channels * 2 + base_channels, base_channels)       # skip from enc1
        
        self.final_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)
        
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
    def _block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(inplace=True)
        )
    
    def forward(self, x, time_emb=None):
        # x shape: [batch, 2, H, W] - конкатенированные LR и noisy HR
        # x[:, 0:1] - LR image, x[:, 1:2] - noisy HR image
        
        # Process time embedding
        if time_emb is not None:
            time_emb = self.time_mlp(time_emb)
            time_emb = time_emb.view(-1, time_emb.shape[1], 1, 1)
        
        # Encoder path (работает с конкатенированными данными)
        e1 = self.enc1(x)  # [batch, 64, H, W]
        e2 = self.enc2(self.pool(e1))  # [batch, 128, H/2, W/2]
        e3 = self.enc3(self.pool(e2))  # [batch, 256, H/4, W/4]
        
        # Bottleneck with time embedding
        bottleneck_input = self.pool(e3)  # [batch, 256, H/8, W/8]
        if time_emb is not None:
            time_emb_expanded = time_emb.expand(-1, -1, bottleneck_input.shape[2], bottleneck_input.shape[3])
            # Добавляем временные эмбеддинги к bottleneck
            bottleneck_input = bottleneck_input + time_emb_expanded[:, :bottleneck_input.shape[1]]
        
        bottleneck = self.bottleneck(bottleneck_input)  # [batch, 512, H/8, W/8]
        
        # Decoder path with skip connections
        d3 = self.dec3(torch.cat([self.upsample(bottleneck), e3], dim=1))  # [batch, 256, H/4, W/4]
        d2 = self.dec2(torch.cat([self.upsample(d3), e2], dim=1))  # [batch, 128, H/2, W/2]
        d1 = self.dec1(torch.cat([self.upsample(d2), e1], dim=1))  # [batch, 64, H, W]
        
        return self.final_conv(d1)  # [batch, 1, H, W] - предсказанный шум

class SuperResolutionDiffusion:
    def __init__(self, model, diffusion, device='cuda', scale_factor=4):
        self.model = model.to(device)
        self.diffusion = diffusion
        self.device = device
        self.scale_factor = scale_factor
        
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=1000)
        
    def get_time_embedding(self, timesteps, dim=32):
        """Создание синусоидальных временных эмбеддингов"""
        half_dim = dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=self.device) * -emb)
        emb = timesteps[:, None].float() * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        if dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb
    
    def prepare_data(self, lr_imgs, hr_imgs):
        """Подготовка пар LR-HR изображений"""
        # Интерполяция LR до размера HR для конкатенации
        lr_upscaled = F.interpolate(lr_imgs, size=hr_imgs.shape[2:], 
                                  mode='bilinear', align_corners=True)
        return lr_upscaled, hr_imgs
    
    def train_step(self, lr_imgs, hr_imgs):
        self.model.train()
        batch_size = lr_imgs.shape[0]
        
        # Подготовка данных
        lr_upscaled, hr_imgs = self.prepare_data(lr_imgs, hr_imgs)
        
        # Sample timesteps
        t = self.diffusion.sample_timesteps(batch_size, self.device)
        
        # Add noise to HR images
        noise = torch.randn_like(hr_imgs)
        noisy_hr = self.diffusion.add_noise(hr_imgs, t, noise)
        
        # Get time embeddings
        time_emb = self.get_time_embedding(t)
        
        # Concatenate upscaled LR and noisy HR as input to model (2 channels)
        model_input = torch.cat([lr_upscaled, noisy_hr], dim=1)  # [batch, 2, H, W]
        
        # Predict noise
        predicted_noise = self.model(model_input, time_emb)
        
        # Simple MSE loss between predicted and actual noise
        loss = F.mse_loss(predicted_noise, noise)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        self.scheduler.step()
        
        return loss.item()
    
    @torch.no_grad()
    def super_resolve(self, lr_imgs, sampling_steps=50):
        """Генерация HR изображения из LR"""
        self.model.eval()
        
        # Определяем целевой размер HR
        if self.scale_factor > 1:
            target_size = (lr_imgs.shape[2] * self.scale_factor, 
                          lr_imgs.shape[3] * self.scale_factor)
        else:
            # Если scale_factor = 1, используем размер LR
            target_size = (lr_imgs.shape[2], lr_imgs.shape[3])
            
        lr_upscaled = F.interpolate(lr_imgs, size=target_size, 
                                  mode='bilinear', align_corners=True)
        
        batch_size = lr_imgs.shape[0]
        
        # Start from pure noise of HR size
        x = torch.randn(batch_size, 1, target_size[0], target_size[1], 
                       device=self.device)
        
        # Sampling steps
        step_size = max(1, self.diffusion.timesteps // sampling_steps)
        timesteps = list(reversed(range(0, self.diffusion.timesteps, step_size)))
        
        for i, t in enumerate(timesteps):
            t_batch = torch.full((batch_size,), t, device=self.device, dtype=torch.long)
            time_emb = self.get_time_embedding(t_batch)
            
            # Model input: concatenated upscaled LR and current noisy HR
            model_input = torch.cat([lr_upscaled, x], dim=1)  # [batch, 2, H, W]
            predicted_noise = self.model(model_input, time_emb)
            
            # DDPM sampling step
            alpha = self.diffusion.alphas.to(self.device)[t_batch].view(-1, 1, 1, 1)
            alpha_cumprod = self.diffusion.alphas_cumprod.to(self.device)[t_batch].view(-1, 1, 1, 1)
            beta = self.diffusion.betas.to(self.device)[t_batch].view(-1, 1, 1, 1)
            
            if t > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
            
            # Denoising step
            x = (1 / torch.sqrt(alpha)) * (
                x - ((1 - alpha) / torch.sqrt(1 - alpha_cumprod)) * predicted_noise
            ) + torch.sqrt(beta) * noise
            
            # Clip values for stability
            if i % 10 == 0:
                x = torch.clamp(x, -1.0, 1.0)
        
        return torch.clamp(x, -1.0, 1.0)

class GaussianDiffusion:
    def __init__(self, timesteps=1000, beta_schedule='linear'):
        self.timesteps = timesteps
        
        if beta_schedule == 'linear':
            beta_start = 1e-4
            beta_end = 0.02
            self.betas = torch.linspace(beta_start, beta_end, timesteps)
        else:
            self.betas = torch.linspace(1e-4, 0.02, timesteps)
            
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        
        # Calculations for diffusion
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        
    def add_noise(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
            
        sqrt_alpha_cumprod = self.sqrt_alphas_cumprod.to(x_start.device)[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_alphas_cumprod.to(x_start.device)[t].view(-1, 1, 1, 1)
        
        return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha_cumprod * noise
    
    def sample_timesteps(self, n, device):
        return torch.randint(low=0, high=self.timesteps, size=(n,), device=device)

# Использование
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Инициализация модели с ПРАВИЛЬНЫМИ каналами
diffusion = GaussianDiffusion(timesteps=1000)
model = CorrectSuperResolutionUNet(in_channels=2, out_channels=1, base_channels=64)  # 2 входных канала!
sr_model = SuperResolutionDiffusion(model, diffusion, device, scale_factor=1)

# Проверка архитектуры
test_input = torch.randn(4, 2, 128, 128).to(device)  # 2 канала!
test_time_emb = torch.randn(4, 32).to(device)
with torch.no_grad():
    test_output = model(test_input, test_time_emb)
print(f"Model test - Input: {test_input.shape}, Output: {test_output.shape}")

# Обучение
num_epochs = 50
for epoch in range(num_epochs):
    epoch_loss = 0
    for i, (lr_imgs, hr_imgs) in enumerate(dataloader):
        lr_imgs = lr_imgs.to(device).float()
        hr_imgs = hr_imgs.to(device).float()
        
        # Нормализация в диапазон [-1, 1]
        lr_imgs = (lr_imgs - 0.5) * 2
        hr_imgs = (hr_imgs - 0.5) * 2
        
        loss = sr_model.train_step(lr_imgs, hr_imgs)
        epoch_loss += loss
        
        if i % 100 == 0:
            print(f'Epoch {epoch}, Batch {i}, Loss: {loss:.4f}')
    
    avg_loss = epoch_loss / len(dataloader)
    print(f'Epoch {epoch} Average Loss: {avg_loss:.4f}')
    
    # Валидация и визуализация каждые 5 эпох
    if epoch % 5 == 0:
        with torch.no_grad():
            test_lr, test_hr = next(iter(dataloader))
            test_lr = test_lr[:4].to(device).float()
            test_hr = test_hr[:4].to(device).float()
            
            test_lr = (test_lr - 0.5) * 2
            test_hr = (test_hr - 0.5) * 2
            
            # Генерация HR
            generated_hr = sr_model.super_resolve(test_lr, sampling_steps=50)
            
            # Денормализация для визуализации
            generated_hr = (generated_hr + 1) / 2
            test_lr_vis = (test_lr + 1) / 2
            test_hr_vis = (test_hr + 1) / 2
            
            # Визуализация
            fig, axes = plt.subplots(4, 3, figsize=(12, 16))
            for idx in range(4):
                axes[idx, 0].imshow(test_lr_vis[idx, 0].cpu(), cmap='viridis')
                axes[idx, 0].set_title('LR Input')
                axes[idx, 1].imshow(generated_hr[idx, 0].cpu(), cmap='viridis')
                axes[idx, 1].set_title('Generated HR')
                axes[idx, 2].imshow(test_hr_vis[idx, 0].cpu(), cmap='viridis')
                axes[idx, 2].set_title('True HR')
            
            plt.tight_layout()
            plt.savefig(f'sr_results_epoch_{epoch}.png')
            plt.close()
            print(f"Saved visualization for epoch {epoch}")

    # Сохранение модели
    if epoch % 10 == 0:
        torch.save(model.state_dict(), f'sr_diffusion_model_epoch_{epoch}.pth')

Using device: cuda
Model test - Input: torch.Size([4, 2, 128, 128]), Output: torch.Size([4, 1, 128, 128])
Epoch 0, Batch 0, Loss: 1.1425
Epoch 0, Batch 100, Loss: 0.1226
Epoch 0, Batch 200, Loss: 0.1663
Epoch 0, Batch 300, Loss: 0.0884
Epoch 0, Batch 400, Loss: 0.0815
Epoch 0, Batch 500, Loss: 0.1083
Epoch 0, Batch 600, Loss: 0.1794
Epoch 0, Batch 700, Loss: 0.2399
Epoch 0, Batch 800, Loss: 0.1759
Epoch 0 Average Loss: 0.1571
Saved visualization for epoch 0
Epoch 1, Batch 0, Loss: 0.1399
Epoch 1, Batch 100, Loss: 0.0907
Epoch 1, Batch 200, Loss: 0.1921
Epoch 1, Batch 300, Loss: 0.1270
Epoch 1, Batch 400, Loss: 0.1455
Epoch 1, Batch 500, Loss: 0.1276
Epoch 1, Batch 600, Loss: 0.1172
Epoch 1, Batch 700, Loss: 0.1296
Epoch 1, Batch 800, Loss: 0.1151
Epoch 1 Average Loss: 0.1356
Epoch 2, Batch 0, Loss: 0.1191
Epoch 2, Batch 100, Loss: 0.0892
Epoch 2, Batch 200, Loss: 0.1357
Epoch 2, Batch 300, Loss: 0.0816
Epoch 2, Batch 400, Loss: 0.1505
Epoch 2, Batch 500, Loss: 0.1175
Epoch 2, Batch 600

KeyboardInterrupt: 

In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from torchmetrics import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio
class MetricsCalculator:
    def __init__(self, device):
        self.device = device
        self.ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
        self.psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)
    
    def calculate_metrics(self, generated, target):
        """Вычисление метрик качества"""
        generated = generated.to(self.device)
        target = target.to(self.device)
        
        # MSE
        mse = F.mse_loss(generated, target)
        
        # PSNR
        psnr_val = self.psnr(generated, target)
        
        # SSIM
        ssim_val = self.ssim(generated, target)
        
        return {
            'mse': mse.item(),
            'psnr': psnr_val.item(),
            'ssim': ssim_val.item()
        }
    
class DDIMSuperResolution:
    def __init__(self, model, diffusion, device='cuda', eta=0.0):
        self.model = model.to(device)
        self.diffusion = diffusion
        self.device = device
        self.eta = eta  # η=0 для детерминированного sampling, η=1 для стохастического
        self.metrics_calculator = MetricsCalculator(device)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=1000)
        
        # Предварительно вычисляем коэффициенты для DDIM
        self.prepare_ddim_coefficients()
    @torch.no_grad()    
    def validate(self, dataloader, num_batches=10, sampling_steps=20):
        """Валидация на части датасета с вычислением метрик"""
        self.model.eval()
        
        total_metrics = {'mse': 0.0, 'psnr': 0.0, 'ssim': 0.0}
        num_samples = 0
        
        for i, (lr_imgs, hr_imgs) in enumerate(dataloader):
            if i >= num_batches:
                break
                
            lr_imgs = lr_imgs.to(self.device).float()
            hr_imgs = hr_imgs.to(self.device).float()
            
            # Нормализация
            lr_imgs = (lr_imgs - 0.5) * 2
            hr_imgs = (hr_imgs - 0.5) * 2
            
            # Генерация
            generated_hr = self.fast_ddim_sample(lr_imgs, sampling_steps)
            
            # Денормализация для метрик
            generated_hr = (generated_hr + 1) / 2
            hr_imgs = (hr_imgs + 1) / 2
            
            # Вычисление метрик
            batch_metrics = self.metrics_calculator.calculate_metrics(generated_hr, hr_imgs)
            
            # Суммируем метрики
            for key in total_metrics:
                total_metrics[key] += batch_metrics[key]
            
            num_samples += lr_imgs.size(0)
        
        # Усредняем метрики
        avg_metrics = {key: total_metrics[key] / num_batches for key in total_metrics}
        
        return avg_metrics
    def prepare_ddim_coefficients(self):
        """Предварительное вычисление коэффициентов для DDIM sampling"""
        timesteps = self.diffusion.timesteps
        
        # Коэффициенты для обратного процесса
        self.ddim_alpha = self.diffusion.alphas_cumprod
        self.ddim_alpha_prev = torch.cat([torch.tensor([1.0]), self.ddim_alpha[:-1]])
        
        # DDIM scaling coefficients
        self.ddim_sigma = self.eta * torch.sqrt(
            (1 - self.ddim_alpha_prev) / (1 - self.ddim_alpha) * 
            (1 - self.ddim_alpha / self.ddim_alpha_prev)
        )
        
        self.ddim_alpha = self.ddim_alpha.to(self.device)
        self.ddim_alpha_prev = self.ddim_alpha_prev.to(self.device)
        self.ddim_sigma = self.ddim_sigma.to(self.device)
    
    def get_time_embedding(self, timesteps, dim=32):
        """Создание синусоидальных временных эмбеддингов"""
        half_dim = dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=self.device) * -emb)
        emb = timesteps[:, None].float() * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        if dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb
    
    def train_step(self, lr_imgs, hr_imgs):
        """Обычный training step как в DDPM"""
        self.model.train()
        batch_size = lr_imgs.shape[0]
        
        # Sample timesteps
        t = self.diffusion.sample_timesteps(batch_size, self.device)
        
        # Add noise to HR images
        noise = torch.randn_like(hr_imgs)
        noisy_hr = self.diffusion.add_noise(hr_imgs, t, noise)
        
        # Get time embeddings
        time_emb = self.get_time_embedding(t)
        
        # Concatenate LR and noisy HR
        model_input = torch.cat([lr_imgs, noisy_hr], dim=1)
        
        # Predict noise
        predicted_noise = self.model(model_input, time_emb)
        
        # Simple MSE loss
        loss = F.mse_loss(predicted_noise, noise)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        self.scheduler.step()
        
        return loss.item()
    
    @torch.no_grad()
    def ddim_sample(self, lr_imgs, sampling_steps=50):
        """DDIM sampling - значительно быстрее чем DDPM"""
        self.model.eval()
        
        batch_size = lr_imgs.shape[0]
        
        # Start from pure noise
        x = torch.randn_like(lr_imgs)
        
        # Создаем расписание timesteps для sampling
        step_size = self.diffusion.timesteps // sampling_steps
        timesteps = list(reversed(range(0, self.diffusion.timesteps, step_size)))
        timesteps_next = [-1] + timesteps[:-1]
        
        for i, (t, t_next) in enumerate(zip(timesteps, timesteps_next)):
            t_batch = torch.full((batch_size,), t, device=self.device, dtype=torch.long)
            time_emb = self.get_time_embedding(t_batch)
            
            # Model input
            model_input = torch.cat([lr_imgs, x], dim=1)
            predicted_noise = self.model(model_input, time_emb)
            
            # DDIM sampling step
            alpha_t = self.ddim_alpha[t]
            alpha_t_prev = self.ddim_alpha_prev[t_next] if t_next >= 0 else torch.tensor(1.0).to(self.device)
            
            # Предсказываем x0
            pred_x0 = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
            
            # Направление к x_t
            dir_xt = torch.sqrt(1 - alpha_t_prev - self.ddim_sigma[t]**2) * predicted_noise
            
            # Добавляем шум если нужно
            if self.eta > 0 and t_next >= 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
                
            # Обновляем x
            x = torch.sqrt(alpha_t_prev) * pred_x0 + dir_xt + self.ddim_sigma[t] * noise
            
            # Clip values for stability
            if i % 10 == 0:
                x = torch.clamp(x, -1.0, 1.0)
        
        return torch.clamp(x, -1.0, 1.0)
    
    @torch.no_grad()
    def fast_ddim_sample(self, lr_imgs, sampling_steps=20):
        """Еще более быстрый DDIM sampling"""
        self.model.eval()
        
        batch_size = lr_imgs.shape[0]
        x = torch.randn_like(lr_imgs)
        
        # Используем меньше шагов
        timesteps = np.linspace(0, self.diffusion.timesteps - 1, sampling_steps, dtype=int)
        timesteps = list(reversed(timesteps))
        
        for i, t in enumerate(timesteps):
            t_batch = torch.full((batch_size,), t, device=self.device, dtype=torch.long)
            time_emb = self.get_time_embedding(t_batch)
            
            model_input = torch.cat([lr_imgs, x], dim=1)
            predicted_noise = self.model(model_input, time_emb)
            
            alpha_t = self.ddim_alpha[t]
            
            if i < len(timesteps) - 1:
                t_next = timesteps[i + 1]
                alpha_t_prev = self.ddim_alpha[t_next]
                
                # Упрощенный DDIM step
                pred_x0 = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
                x = torch.sqrt(alpha_t_prev) * pred_x0 + torch.sqrt(1 - alpha_t_prev) * predicted_noise
            else:
                # Последний шаг - просто предсказываем x0
                x = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
        
        return torch.clamp(x, -1.0, 1.0)

class SuperResolutionUNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, base_channels=64, time_emb_dim=32):
        super().__init__()
        
        self.time_emb_dim = time_emb_dim
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, base_channels * 4),
            nn.SiLU(),
            nn.Linear(base_channels * 4, base_channels * 4)
        )
        
        # Encoder
        self.enc1 = self._block(in_channels, base_channels)
        self.enc2 = self._block(base_channels, base_channels * 2)
        self.enc3 = self._block(base_channels * 2, base_channels * 4)
        
        # Bottleneck
        self.bottleneck = self._block(base_channels * 4, base_channels * 8)
        
        # Decoder 
        self.dec3 = self._block(base_channels * 8 + base_channels * 4, base_channels * 4)
        self.dec2 = self._block(base_channels * 4 + base_channels * 2, base_channels * 2)
        self.dec1 = self._block(base_channels * 2 + base_channels, base_channels)
        
        self.final_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)
        
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
    def _block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(inplace=True)
        )
    
    def forward(self, x, time_emb=None):
        # Process time embedding
        if time_emb is not None:
            time_emb = self.time_mlp(time_emb)
            time_emb = time_emb.view(-1, time_emb.shape[1], 1, 1)
        
        # Encoder path
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        
        # Bottleneck with time embedding
        bottleneck_input = self.pool(e3)
        if time_emb is not None:
            time_emb_expanded = time_emb.expand(-1, -1, bottleneck_input.shape[2], bottleneck_input.shape[3])
            bottleneck_input = bottleneck_input + time_emb_expanded[:, :bottleneck_input.shape[1]]
        
        bottleneck = self.bottleneck(bottleneck_input)
        
        # Decoder path with skip connections
        d3 = self.dec3(torch.cat([self.upsample(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.upsample(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.upsample(d2), e1], dim=1))
        
        return self.final_conv(d1)

class GaussianDiffusion:
    def __init__(self, timesteps=1000, beta_schedule='linear'):
        self.timesteps = timesteps
        
        if beta_schedule == 'linear':
            beta_start = 1e-4
            beta_end = 0.02
            self.betas = torch.linspace(beta_start, beta_end, timesteps)
        else:
            self.betas = torch.linspace(1e-4, 0.02, timesteps)
            
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        
        # Calculations for diffusion
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        
    def add_noise(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
            
        sqrt_alpha_cumprod = self.sqrt_alphas_cumprod.to(x_start.device)[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_alphas_cumprod.to(x_start.device)[t].view(-1, 1, 1, 1)
        
        return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha_cumprod * noise
    
    def sample_timesteps(self, n, device):
        return torch.randint(low=0, high=self.timesteps, size=(n,), device=device)

# Использование
device = 'cuda' if torch.cuda.is_available() else 'cpu'
best_psnr = 0
# Инициализация с DDIM
diffusion = GaussianDiffusion(timesteps=1000)
model = SuperResolutionUNet(in_channels=2, out_channels=1, base_channels=64)
ddim_sr = DDIMSuperResolution(model, diffusion, device, eta=0.0)  # η=0 для детерминированного sampling

# Обучение (такое же как раньше)
for epoch in range(num_epochs):
    epoch_loss = 0
    for i, (lr_imgs, hr_imgs) in enumerate(dataloader):
        lr_imgs = lr_imgs.to(device).float()
        hr_imgs = hr_imgs.to(device).float()
        
        # Нормализация в диапазон [-1, 1]
        lr_imgs = (lr_imgs - 0.5) * 2
        hr_imgs = (hr_imgs - 0.5) * 2
        
        loss = ddim_sr.train_step(lr_imgs, hr_imgs)
        epoch_loss += loss
        
        if i % 100 == 0:
            print(f'Epoch {epoch}, Batch {i}, Loss: {loss:.4f}')
    
    print(f'Epoch {epoch} Average Loss: {epoch_loss/len(dataloader):.4f}')
    
        # Валидация и метрики
    val_metrics = ddim_sr.validate(dataloader, num_batches=5, sampling_steps=20)
    
    print(f'Epoch {epoch} Validation Metrics:')
    print(f'  MSE: {val_metrics["mse"]:.6f}')
    print(f'  PSNR: {val_metrics["psnr"]:.2f} dB')
    print(f'  SSIM: {val_metrics["ssim"]:.4f}')
        # Сохранение лучшей модели
    if val_metrics['psnr'] > best_psnr:
        best_psnr = val_metrics['psnr']
        torch.save(model.state_dict(), f'best_ddim_sr_model{epoch}.pth')
        print(f'New best model saved with PSNR: {best_psnr:.2f} dB')

Epoch 0, Batch 0, Loss: 1.1508
Epoch 0, Batch 100, Loss: 0.1828
Epoch 0, Batch 200, Loss: 0.1629
Epoch 0, Batch 300, Loss: 0.1063
Epoch 0, Batch 400, Loss: 0.1217
Epoch 0, Batch 500, Loss: 0.1123
Epoch 0, Batch 600, Loss: 0.1267
Epoch 0, Batch 700, Loss: 0.1419
Epoch 0, Batch 800, Loss: 0.1330
Epoch 0 Average Loss: 0.1582
Epoch 0 Validation Metrics:
  MSE: 0.415078
  PSNR: 3.82 dB
  SSIM: 0.0055
New best model saved with PSNR: 3.82 dB
Epoch 1, Batch 0, Loss: 0.1303
Epoch 1, Batch 100, Loss: 0.2450
Epoch 1, Batch 200, Loss: 0.1046
Epoch 1, Batch 300, Loss: 0.1836
Epoch 1, Batch 400, Loss: 0.1492
Epoch 1, Batch 500, Loss: 0.0733
Epoch 1, Batch 600, Loss: 0.0809
Epoch 1, Batch 700, Loss: 0.1572
Epoch 1, Batch 800, Loss: 0.1403
Epoch 1 Average Loss: 0.1360
Epoch 1 Validation Metrics:
  MSE: 0.468859
  PSNR: 3.29 dB
  SSIM: 0.0094
Epoch 2, Batch 0, Loss: 0.1886
Epoch 2, Batch 100, Loss: 0.0932
Epoch 2, Batch 200, Loss: 0.1511
Epoch 2, Batch 300, Loss: 0.1252
Epoch 2, Batch 400, Loss: 0.1886

KeyboardInterrupt: 